# 09a - Do the multimodal features make PLOS predictable on Synthea? Whole stay vs first 24 hours (run_02)

Real data only, no synthetic rows. Same classifier, same patient split (train / val / test from notebook 01).
Compares average precision (AP) on test for:
- the tabular features used so far (notebook 01),
- the 524-number vector from notebook 07, block by block (profile, conditions, procedures, notes, medications, flags),
- everything combined.

If AP rises clearly above the tabular baseline, Synthea becomes usable for measuring augmentation gain on these features.
**Caution:** notes, medications, procedures, vitals and the activity flags come from DURING the stay. A large jump may
partly be information that is not available at admission. Cell 4 reports an admission-safe subset separately.

**Versions compared:** `features_full` (whole stay) and `features_full_24h` (first 24 hours, built by notebook 07
with `WINDOW_HOURS = 24`). If the 24-hour version keeps most of the gain, the signal is available at admission.
If it collapses, the whole-stay gain was mostly the stay's own length leaking into the features.

Runtime: CPU, about 10-15 minutes.

In [ ]:
# Cell 1 - setup
import os, sys, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
if "google.colab" in sys.modules:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
VERSIONS = [v for v in ["features_full", "features_full_24h"] if os.path.exists(f"{BASE}/{v}/encounters_524.parquet")]
meta = json.load(open(f"{BASE}/genai/meta.json"))
tab = pd.concat([pd.read_parquet(f"{BASE}/genai/{s}.parquet") for s in ("train", "val", "test")])
TAB_COLS = meta["num_cols"] + meta["cat_cols"] + meta["bin_cols"]
NB = int(os.environ.get("NBOOT", 1000))
print("Feature versions found:", VERSIONS)

In [ ]:
# Cell 2 - one function: every feature set for one version, AP on test with paired patient bootstrap vs tabular
def tab_X(d): return pd.get_dummies(d[TAB_COLS], columns=meta["cat_cols"], dtype=float)
def X_of(d, kind, cols):
    if kind == "tab": return tab_X(d)
    if kind == "cols": return d[cols].astype(float)
    t = tab_X(d).reset_index(drop=True)
    return pd.concat([d[cols].astype(float).reset_index(drop=True), t.drop(columns=[c for c in t if c in cols])], axis=1)
def fit_pred(tr, va, te, kind, cols):
    Xtr = X_of(tr, kind, cols)
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(Xtr, tr.is_plos)
    f = lambda d: m.predict_proba(X_of(d, kind, cols).reindex(columns=Xtr.columns, fill_value=0))[:, 1]
    return Xtr.shape[1], f(va), f(te)

def run_version(v):
    F = pd.read_parquet(f"{BASE}/{v}/encounters_524.parquet"); fm = json.load(open(f"{BASE}/{v}/features_meta.json"))
    D = F.merge(tab[["hadm_id"] + [c for c in TAB_COLS if c not in F.columns]], on="hadm_id", how="left")
    B, PROFILE, FLAGS = fm["blocks"], fm["profile"], ["lab_active", "med_active"]
    SETS = {"A. tabular": ("tab", None), "B. profile + vitals": ("cols", PROFILE),
            "C. B + conditions": ("cols", PROFILE + B["conditions"]), "D. B + procedures": ("cols", PROFILE + B["procedures"]),
            "E. B + notes": ("cols", PROFILE + B["notes"]), "F. B + medications": ("cols", PROFILE + B["medications"]),
            "G. B + activity flags": ("cols", PROFILE + FLAGS), "H. full 524": ("cols", fm["feat524"]),
            "I. full 524 + tabular": ("both", fm["feat524"])}
    tr, va, te = (D[D.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    yte = te.is_plos.values; rng = np.random.default_rng(0)
    pats = te.subject_id.values; up = np.unique(pats); pos = {p: np.where(pats == p)[0] for p in up}
    boots = [np.concatenate([pos[p] for p in rng.choice(up, len(up))]) for _ in range(NB)]
    preds, rows = {}, []
    for name, (kind, cols) in SETS.items():
        n, pv, pt = fit_pred(tr, va, te, kind, cols); preds[name] = pt
        rows.append({"features": name, "n_features": n, "AP_val": average_precision_score(va.is_plos, pv),
                     "AP_test": average_precision_score(yte, pt), "AUROC_test": roc_auc_score(yte, pt)})
    base = preds["A. tabular"]
    for r in rows:
        g = [average_precision_score(yte[i], preds[r["features"]][i]) - average_precision_score(yte[i], base[i])
             for i in boots if yte[i].sum() > 0]
        r["AP_gain_vs_A"] = r["AP_test"] - average_precision_score(yte, base)
        r["ci_low"], r["ci_high"] = np.percentile(g, [2.5, 97.5])
    SAFE = ["anchor_age", "gender_enc", "race_enc", "ethnicity_enc"]
    STAY = {"notes": B["notes"], "medications": B["medications"] + ["med_active"], "procedures": B["procedures"],
            "vitals+labs": [c for c in PROFILE if c.startswith("vital_")] + ["has_vitals", "lab_active"]}
    ab = []
    for name, cols in [("demographics only", SAFE)] + [(f"+ {k}", SAFE + c) for k, c in STAY.items()]:
        _, _, pt = fit_pred(tr, va, te, "cols", cols)
        ab.append({"features": name, "AP_test": average_precision_score(yte, pt)})
    return pd.DataFrame(rows).set_index("features"), pd.DataFrame(ab).set_index("features"), yte.mean()

In [ ]:
# Cell 3 - run every version and save
results = {}
for v in VERSIONS:
    res, ab, prev = run_version(v); results[v] = (res, ab)
    res.to_csv(f"{BASE}/{v}/09a_feature_utility.csv"); ab.to_csv(f"{BASE}/{v}/09a_block_ablation.csv")
    print(f"\n=== {v} === (test PLOS prevalence {prev:.3f})")
    print(res.round(4).to_string())

In [ ]:
# Cell 4 - the decisive comparison: whole stay vs first 24 hours
if len(results) == 2:
    a, b = results["features_full"], results["features_full_24h"]
    cmp = pd.DataFrame({"AP whole stay": a[0]["AP_test"], "AP first 24 h": b[0]["AP_test"]})
    cmp["share of gain kept"] = (cmp["AP first 24 h"] - cmp.loc["A. tabular", "AP first 24 h"]) / \
                                (cmp["AP whole stay"] - cmp.loc["A. tabular", "AP whole stay"]).replace(0, np.nan)
    blk = pd.DataFrame({"AP whole stay": a[1]["AP_test"], "AP first 24 h": b[1]["AP_test"]})
    print("Feature sets:"); print(cmp.round(4).to_string())
    print("\nDemographics + one block at a time:"); print(blk.round(4).to_string())
    cmp.to_csv(f"{BASE}/features_full_24h/09a_whole_vs_24h.csv")
else:
    print("Run notebook 07 with WINDOW_HOURS = 24 first to create features_full_24h.")